# Practical 7 – Positional Encoding and Layer Normalization

## Objective
Implement positional encoding and Layer Normalization and demonstrate their use inside a simple Transformer block.

## Requirements
Python, PyTorch, Matplotlib

## 1. Import libraries

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## 2. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

d_model = 16
positional_encoding = PositionalEncoding(d_model)

x = torch.zeros(1, 20, d_model)
encoded = positional_encoding(x)

print("Input shape :", x.shape)
print("Output shape:", encoded.shape)

## 3. Visualize positional encoding

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(encoded[0].detach().numpy().T, aspect="auto")
plt.xlabel("Position")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.show()

## 4. Layer Normalization

In [ ]:
layer_norm = nn.LayerNorm(d_model)

normalized = layer_norm(encoded)

print("Mean before normalization:",
      encoded.mean().item())
print("Standard deviation before normalization:",
      encoded.std().item())

print("\nMean after LayerNorm:",
      normalized.mean().item())
print("Standard deviation after LayerNorm:",
      normalized.std().item())

## 5. Transformer block example

In [ ]:
class SimpleTransformerBlock(nn.Module):
    def __init__(self, d_model=16, num_heads=4):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attention_output, attention_weights = self.attention(x, x, x)

        x = self.norm1(x + attention_output)

        ff_output = self.feed_forward(x)
        x = self.norm2(x + ff_output)

        return x, attention_weights

block = SimpleTransformerBlock()

output, attention = block(encoded)

print("Transformer block output:", output.shape)
print("Attention matrix:", attention.shape)

## Result
Positional encoding provides position information to a Transformer, while Layer Normalization stabilizes the representations inside the network.